In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast, AutoTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import os
import spacy
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
import re
from collections import defaultdict
from collections import Counter

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base Model

# Read Data

In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered.csv')

# Create Training and Test Sets

In [ ]:
train_text, temp_text, train_labels, temp_labels = train_test_split(df['full_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

In [ ]:
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

tokens_test = tokenizer.batch_encode_plus(
    test_text.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

In [ ]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

/tmp/ipython-input-101-2686642901.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_seq = torch.tensor(tokens_train['input_ids'])
/tmp/ipython-input-101-2686642901.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_mask = torch.tensor(tokens_train['attention_mask'])
/tmp/ipython-input-101-2686642901.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_seq = torch.tensor(tokens_val['input_ids'])
/tmp/ipython-input-101-2686642901.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clon

In [ ]:
batch_size = 16
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [ ]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


In [ ]:
# Unfreeze top 2 layers to start (layers 4 and 5)
for i in [4, 5]:
    for param in bert.transformer.layer[i].parameters():
        param.requires_grad = True


# Create Model

In [ ]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(bert.config.hidden_size,128)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(128,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [ ]:
model = BERT_Arch(bert)
model = model.to(device)

# Test Model

In [ ]:
model = torch.load('../saved_models/explicit_gender_token_removal/distillbert.pt', weights_only=False)

In [ ]:
model.eval()  # Set model to eval mode

# Create DataLoader for test set
test_data = TensorDataset(test_seq, test_mask, test_y)
test_dataloader = DataLoader(test_data, batch_size=32)  # adjust batch size as needed

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        sent_id, mask, labels = [b.to(device) for b in batch]

        # Forward pass
        outputs = model(sent_id, mask)  # shape: (batch_size, num_classes)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.46      0.40      0.43       279
           1       0.74      0.79      0.76       620

    accuracy                           0.67       899
   macro avg       0.60      0.59      0.60       899
weighted avg       0.65      0.67      0.66       899

Test Confusion Matrix: 
 [[111 168]
 [132 488]]


In [ ]:
preds_full = all_preds.copy()
preds_full = np.array(preds_full)

# Model With Implicitly Gendered Tokens Removed

# Read Data

In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered.csv')

In [ ]:
top_tokens = pd.read_csv('../data/shap_tokens/top_tokens_shap_distillbert.csv')

In [ ]:
tokens_to_remove = set(top_tokens["token"].str.lower())

# pattern = (
#     r"(?:^|\b|[^\w\s])"                              # start of string or word boundary
#     + r"(?P<token>" + "|".join(re.escape(t) for t in tokens_to_remove) + r")"  # the tokens
#     + r"(?P<suffix>'s|’s)?"                          # optional possessive suffix
#     + r"(?=\b|[^\w\s]|$)"                            # lookahead for word boundary or punctuation
# )

# def remove_tokens_regex(text, pattern):
#     cleaned = re.sub(pattern, '', text, flags=re.IGNORECASE)
#     cleaned = re.sub(r'\s+', ' ', cleaned).strip()
#     return cleaned

# df["full_text"] = df["full_text"].apply(lambda x: remove_tokens_regex(x, pattern))

import pandas as pd
from transformers import AutoTokenizer
import re

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
mask_token = tokenizer.mask_token  # '<mask>'

# Token list
token_list = top_tokens["token"].tolist()

# Escape special regex characters
escaped_tokens = [re.escape(token) for token in token_list]
pattern = "(" + "|".join(escaped_tokens) + ")"

# Replace matches with spaced mask token
def mask_text(text):
    return re.sub(pattern, f" {mask_token} ", text)

# Apply the masking
df["full_text"] = df["full_text"].apply(mask_text)


# Create Training and Test Sets

In [ ]:
train_text, temp_text, train_labels, temp_labels = train_test_split(df['full_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

val_text, test_text_debiased, val_labels, test_labels_debiased = train_test_split(temp_text, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

In [ ]:
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

tokens_test = tokenizer.batch_encode_plus(
    test_text_debiased.tolist(),
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

In [ ]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels_debiased.tolist())

/tmp/ipython-input-117-3367856088.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_seq = torch.tensor(tokens_train['input_ids'])
/tmp/ipython-input-117-3367856088.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_mask = torch.tensor(tokens_train['attention_mask'])
/tmp/ipython-input-117-3367856088.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_seq = torch.tensor(tokens_val['input_ids'])
/tmp/ipython-input-117-3367856088.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clon

In [ ]:
batch_size = 16
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [ ]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


In [ ]:
# Unfreeze top 2 layers to start (layers 4 and 5)
for i in [4, 5]:
    for param in bert.transformer.layer[i].parameters():
        param.requires_grad = True


# Create Model

In [ ]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(bert.config.hidden_size,128)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(128,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [ ]:
model = BERT_Arch(bert)
model = model.to(device)

# Test Model

In [ ]:
model = torch.load('../saved_models/implicit_gender_token_removal/saved_model_shap_tokens_removed_distillbert.pt', weights_only=False)


In [ ]:
model.eval()  # Set model to eval mode

# Create DataLoader for test set
test_data = TensorDataset(test_seq, test_mask, test_y)
test_dataloader = DataLoader(test_data, batch_size=32)  # adjust batch size as needed

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        sent_id, mask, labels = [b.to(device) for b in batch]

        # Forward pass
        outputs = model(sent_id, mask)  # shape: (batch_size, num_classes)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.40      0.43      0.41       279
           1       0.73      0.71      0.72       620

    accuracy                           0.62       899
   macro avg       0.57      0.57      0.57       899
weighted avg       0.63      0.62      0.63       899

Test Confusion Matrix: 
 [[119 160]
 [178 442]]


In [ ]:
preds_debiased = all_preds.copy()
preds_debiased = np.array(preds_debiased)

# Compare Output

In [ ]:
all_labels = np.array(all_labels)

In [ ]:
correct_full = np.where(preds_full == all_labels)[0]
incorrect_debiased = np.where(preds_debiased != all_labels)[0]
interesting_indices = np.intersect1d(correct_full, incorrect_debiased)

In [ ]:
interesting_indices

array([  0,  21,  28,  30,  32,  39,  40,  42,  45,  52,  53,  69,  76,
        83,  86,  87,  89,  96, 101, 108, 111, 117, 124, 137, 138, 144,
       159, 169, 183, 186, 199, 215, 225, 229, 239, 243, 259, 278, 288,
       317, 319, 327, 328, 329, 332, 333, 335, 340, 344, 356, 358, 363,
       366, 392, 418, 423, 439, 446, 462, 463, 466, 468, 483, 484, 485,
       494, 500, 503, 507, 510, 514, 520, 524, 528, 533, 537, 541, 543,
       546, 552, 574, 577, 580, 588, 597, 599, 600, 609, 610, 616, 617,
       623, 627, 633, 640, 649, 657, 660, 664, 665, 668, 684, 693, 697,
       698, 706, 715, 716, 745, 746, 750, 751, 752, 763, 764, 774, 786,
       788, 805, 818, 832, 834, 840, 844, 847, 850, 851, 852, 859, 867,
       869, 871, 876, 884, 898])

In [ ]:
full_texts = pd.DataFrame(test_text).iloc[interesting_indices].reset_index(drop=True)
debiased_texts = pd.DataFrame(test_text_debiased).iloc[interesting_indices].reset_index(drop=True)
true_labels = pd.DataFrame(test_labels).iloc[interesting_indices].reset_index(drop=True)

def find_removed_tokens(original, debiased, tokens_to_remove):
    original_tokens = set(original.lower().split())
    debiased_tokens = set(debiased.lower().split())
    removed = [tok for tok in tokens_to_remove if tok in original_tokens and tok not in debiased_tokens]
    return removed if original != debiased else []

removed_tokens = [
    find_removed_tokens(orig, deb, tokens_to_remove)
    for orig, deb in zip(full_texts["full_text"], debiased_texts["full_text"])
]

result_df = pd.DataFrame({
    "original_text": full_texts["full_text"],
    "debiased_text": debiased_texts["full_text"],
    "removed_tokens": [", ".join(toks) if toks else "" for toks in removed_tokens],
    "true_label": true_labels['label']
})

In [ ]:
result_df['removed_tokens']

,removed_tokens
0,infectious
1,"women, united"
2,impressed
3,
4,"remained, reliable"
...,...
130,humble
131,
132,impressed
133,impressed


In [ ]:
# result_df.to_csv("../data/prediction_comparison_shap_tokens_removed.csv", index=False)